In [ ]:
from discovery_utils.utils import search
from discovery_utils.getters import gtr

from discovery_utils import PROJECT_DIR
VECTOR_DB_DIR = PROJECT_DIR / 'tmp/vector_db'

GTR = gtr.GtrGetter(vector_db_path=VECTOR_DB_DIR)

In [ ]:
Search = search.SearchDataset(GTR, GTR.projects_enriched, "config.yaml")

In [ ]:
search_df = Search.do_search()

In [87]:
from discovery_utils.utils import (
    analysis,
    analysis_gtr,
)
import importlib
importlib.reload(analysis);
importlib.reload(analysis_gtr);

In [ ]:
df = (
    search_df
    .merge(GTR.get_projects_text(), on='id', how='left')
    .query("_score_avg > 0.3")
)
print(len(df))
df_dedup = analysis_gtr.deduplicate_projects(df, description_column='text')
print(len(df_dedup))

In [ ]:
analysis_gtr.funding_per_period(df, period='year', min_year=2010, max_year=2024)

In [ ]:
ts_df = analysis_gtr.get_timeseries(df, period='year', min_year=2010, max_year=2024)
analysis.magnitude_growth(ts_df, 2019, 2024)
# ts_df

In [ ]:
from discovery_utils.getters import crunchbase
CB = crunchbase.CrunchbaseGetter(vector_db_path=VECTOR_DB_DIR)

In [ ]:
SearchCB = search.SearchDataset(CB, CB.organisations_enriched, "config.yaml")
search_cb_df = SearchCB.do_search()

In [ ]:
orgs_df = CB.organisations_enriched.copy()
funds_df = CB.funding_rounds_enriched.copy()

In [391]:
importlib.reload(crunchbase);
CB = crunchbase.CrunchbaseGetter(vector_db_path=VECTOR_DB_DIR)
CB._organisations_enriched = orgs_df
CB._funding_rounds_enriched = funds_df

2024-11-20 16:59:36,544 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss


2024-11-20 16:59:36,660 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2024-11-18


In [116]:
df = (
    search_cb_df
    .query("_score_avg > 0.3")
)

In [204]:
from discovery_utils.utils import (
    analysis,
    analysis_crunchbase,
)
importlib.reload(analysis);
importlib.reload(analysis_crunchbase);

In [ ]:
analysis_crunchbase.orgs_founded_per_period(df, 'year', 2010, 2024)

In [133]:
matching_ids = df.id.to_list()

In [134]:
selected_funding_df = (
    CB.funding_rounds_enriched
    .query("org_id in @matching_ids")
    .query(f"year >= {2010}")
    .query(f"year <= {2024}")
    # .query(f"investment_type in @include_deals")
    .drop_duplicates("funding_round_id")
)

In [191]:
importlib.reload(analysis_crunchbase);

In [ ]:
analysis_crunchbase

In [ ]:
deals_df, deal_counts_df = analysis_crunchbase.get_funding_by_year_and_range(selected_funding_df, 2014, 2024)
deals_df

In [ ]:
deal_counts_df

In [ ]:
(
    selected_funding_df
    .query("announced_on >= '2015-01-01'")
    .query("announced_on < '2016-01-01'")
    .raised_amount_gbp.sum()
)

In [276]:
importlib.reload(analysis_crunchbase);
ts_df = analysis_crunchbase.get_timeseries(df, selected_funding_df, 'year', 2010, 2024)

In [293]:
ts_df

,time_period,year,n_rounds,raised_amount_usd_total,raised_amount_gbp_total,n_orgs_founded
0,2010-01-01,2010,5,3.073000,1.929978,5
1,2011-01-01,2011,6,5.775000,3.608702,2
2,2012-01-01,2012,5,1.467900,0.925512,4
3,2013-01-01,2013,2,0.015000,0.009968,10
4,2014-01-01,2014,7,2.890469,1.702734,14
5,2015-01-01,2015,6,3.067977,2.020709,5
6,2016-01-01,2016,5,3.912430,2.742130,6
7,2017-01-01,2017,3,0.734822,0.565402,7
8,2018-01-01,2018,3,13.547525,10.198627,6
9,2019-01-01,2019,2,0.000000,0.000000,7


In [ ]:
analysis.magnitude_growth(ts_df, 2019, 2024)

In [285]:
importlib.reload(analysis_crunchbase);

In [ ]:
aggregated_funding_types_df = analysis_crunchbase.aggregate_by_funding_round_types(selected_funding_df)
analysis_crunchbase.chart_investment_types(aggregated_funding_types_df)

In [ ]:
analysis_crunchbase.chart_deal_sizes(deals_df)

In [303]:
analysis_crunchbase.chart_deal_sizes_counts(deal_counts_df)

alt.Chart(...)

In [320]:
from discovery_utils.utils import charts
importlib.reload(charts);


In [295]:
ts_df.head(5)

,time_period,year,n_rounds,raised_amount_usd_total,raised_amount_gbp_total,n_orgs_founded
0,2010-01-01,2010,5,3.073000,1.929978,5
1,2011-01-01,2011,6,5.775000,3.608702,2
2,2012-01-01,2012,5,1.467900,0.925512,4
3,2013-01-01,2013,2,0.015000,0.009968,10
4,2014-01-01,2014,7,2.890469,1.702734,14


In [327]:
charts.ts_bar_incomplete(
    ts_df,
    variable='raised_amount_gbp_total',
    variable_title='Funding',
    max_complete_year=2022
)

alt.LayerChart(...)

In [330]:
orgs_to_narrow_categories_df = CB.organisation_categories.explode('category_list')

In [332]:
orgs_to_narrow_categories_df.head(5)

,id,category_list
0,e1393508-30ea-8a36-3f96-dd3226033abd,Publishing
0,e1393508-30ea-8a36-3f96-dd3226033abd,Social Media
0,e1393508-30ea-8a36-3f96-dd3226033abd,Social Media Management
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Cloud Computing
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Collaboration


In [333]:
_df = (
    orgs_to_narrow_categories_df
    .merge(CB.group_to_categories, left_on='category_list', right_on='category')
)

In [336]:
CB.group_to_categories

,group,category
627,Administrative Services,Archiving Service
658,Administrative Services,Call Center
676,Administrative Services,Collection Agency
55,Administrative Services,College Recruiting
592,Administrative Services,Courier Service
...,...,...
263,Video,TV Production
18,Video,Video
300,Video,Video Editing
161,Video,Video Streaming


In [335]:
orgs_to_narrow_categories_df.merge()

,id,category_list
0,e1393508-30ea-8a36-3f96-dd3226033abd,Publishing
0,e1393508-30ea-8a36-3f96-dd3226033abd,Social Media
0,e1393508-30ea-8a36-3f96-dd3226033abd,Social Media Management
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Cloud Computing
1,bf4d7b0e-b34d-2fd8-d292-6049c4f7efc7,Collaboration
...,...,...
3682528,054f83d7-d698-4b34-b4ec-293c8648b695,E-Commerce
3682528,054f83d7-d698-4b34-b4ec-293c8648b695,Marketplace
3682529,751cff94-be30-48c8-b2d1-58c8255ece82,Construction
3682530,51f6584f-5289-40fe-8d76-072007eb4c08,NaN


In [349]:
df = CB.get_companies_in_categories(['Family'])


In [351]:
CB.unique_funding_round_types

['angel',
 'convertible_note',
 'corporate_round',
 'debt_financing',
 'equity_crowdfunding',
 'grant',
 'initial_coin_offering',
 'non_equity_assistance',
 'post_ipo_debt',
 'post_ipo_equity',
 'post_ipo_secondary',
 'pre_seed',
 'private_equity',
 'product_crowdfunding',
 'secondary_market',
 'seed',
 'series_a',
 'series_b',
 'series_c',
 'series_d',
 'series_e',
 'series_f',
 'series_g',
 'series_h',
 'series_i',
 'series_j',
 'series_unknown',
 'undisclosed']

In [352]:
funding_rounds_df = CB.select_funding_rounds(
    org_ids=df.id.to_list(),
    funding_round_types=['angel', 'seed', 'pre_seed', 'series_a']
)

In [356]:
importlib.reload(analysis_crunchbase);

In [358]:
ts_df = analysis_crunchbase.get_timeseries(df, funding_rounds_df, 'year', 2014, 2024)

In [359]:
charts.ts_bar(
    ts_df,
    "raised_amount_gbp_total",
    "Funding",
)

alt.Chart(...)

In [365]:
CB.group_to_categories.category.unique()

array(['Archiving Service', 'Call Center', 'Collection Agency',
       'College Recruiting', 'Courier Service', 'Debt Collections',
       'Delivery', 'Document Preparation', 'Employee Benefits',
       'Extermination Service', 'Facilities Support Services',
       'Housekeeping Service', 'Human Resources', 'Knowledge Management',
       'Office Administration', 'Packaging Services', 'Physical Security',
       'Project Management', 'Staffing Agency', 'Trade Shows',
       'Virtual Workforce', 'Ad Exchange', 'Ad Network', 'Ad Retargeting',
       'Ad Server', 'Ad Targeting', 'Advertising',
       'Advertising Platforms', 'Affiliate Marketing',
       'Local Advertising', 'Mobile Advertising', 'Native Advertising',
       'Outdoor Advertising', 'SEM', 'Social Media Advertising',
       'Video Advertising', 'AgTech', 'Agriculture', 'Animal Feed',
       'Aquaculture', 'Equestrian', 'Farming', 'Forestry', 'Horticulture',
       'Hydroponics', 'Livestock', 'App Discovery', 'Apps',
       '

In [383]:
CB.group_vectors

2024-11-20 16:56:59,904 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/Crunchbase_2024-11-18/category_groups.parquet
2024-11-20 16:57:00,025 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: data/crunchbase/Crunchbase_2024-11-18/category_groups.parquet
2024-11-20 16:57:00,031 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2024-11-20 16:57:00,892 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: m

,group,vector
0,Administrative Services,"[-0.0789601, -0.057139844, -0.015132315, 0.009..."
1,Advertising,"[-0.06080457, 0.00965797, -0.0334141, 0.022970..."
2,Agriculture and Farming,"[0.011781091, 0.0478218, -0.038464304, 0.02491..."
3,Apps,"[-0.051812097, -0.03198021, 0.0066298256, -0.0..."
4,Artificial Intelligence (AI),"[-0.03369453, 0.0392624, -0.002034516, 0.05440..."
5,Biotechnology,"[-0.05557861, -0.007920985, -0.07175106, -0.06..."
6,Blockchain and Cryptocurrency,"[-0.0159732, 0.06255727, -0.066419, 0.01064516..."
7,Clothing and Apparel,"[-0.02693445, 0.07989401, 0.034079853, 0.05068..."
8,Commerce and Shopping,"[0.012635123, 0.013148435, -0.041306533, 0.012..."
9,Community and Lifestyle,"[0.029243192, 0.008376386, -0.05121899, 0.0815..."


In [395]:
CB.find_similar_categories("cyber safety", category_type='narrow')

Batches: 100%|██████████| 1/1 [00:00<00:00, 16.49it/s]


,category,similarity
453,Cyber Security,0.782012
608,Security,0.583227
467,Network Security,0.557836
16,Physical Security,0.529581
370,Public Safety,0.522218
607,Privacy,0.467996
451,Cloud Security,0.454629
484,Internet,0.449748
368,National Security,0.436531
474,Technical Support,0.415475


In [368]:
unique_groups = CB.group_to_categories.group.unique()
vectors = CB.embedding_model.encode(unique_groups)

Batches: 100%|██████████| 2/2 [00:00<00:00,  5.92it/s]


In [371]:
len(unique_groups)

49

In [377]:
len(list(vectors))

49

In [379]:
import pandas as pd
vectors_df = pd.DataFrame(
    data={"group": unique_groups, "vector": list(vectors)}
)

In [380]:
query_embedding = CB.embedding_model.encode(["Cybersecurity"])[0]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.34it/s]


In [381]:
import numpy as np
(
    vectors_df            
    .assign(similarity = vectors_df.vector.apply(lambda x: np.dot(query_embedding, x)))
    .sort_values("similarity", ascending=False)
    .drop(columns="vector")
    .head(10)
)

,group,similarity
37,Privacy and Security,0.508416
24,Information Technology,0.434857
25,Internet Services,0.424183
43,Software,0.387372
4,Artificial Intelligence (AI),0.377659
0,Administrative Services,0.373829
5,Biotechnology,0.357013
41,Science and Engineering,0.336290
22,Hardware,0.316858
13,Data and Analytics,0.311445


In [396]:
orgs_cyber_df = CB.get_companies_in_categories(["Cyber Security"])

In [400]:
orgs_parenting_df = CB.get_companies_in_categories(["Family", "Parenting", "Children", "Teenagers", "Child Care"])

In [401]:
matching_ids = set(orgs_cyber_df.id.to_list()).intersection(orgs_parenting_df.id.to_list())

In [402]:
len(matching_ids)

12

In [420]:
importlib.reload(charts)

<module 'discovery_utils.utils.charts' from '/Users/karlis.kanders/Code/discovery_utils/discovery_utils/utils/charts.py'>

In [404]:
# CB.organisations_enriched.query("id in @matching_ids")

In [417]:
ts_test = pd.concat([
    ts_df.assign(category="Cyber Security"),
    ts_df.assign(category="Parenting").assign(raised_amount_gbp_total=lambda df: df.raised_amount_gbp_total * 0.5)
], ignore_index=True)

In [428]:
(
    charts.configure_plots(
        charts.ts_bar_incomplete(
        ts_test,
        "raised_amount_gbp_total",
        "Funding",
        categories_to_show=["Cyber Security", "Parenting"],
        category_column="category",
        max_complete_year=2022
    ), "Ahahaha")
)




alt.LayerChart(...)